# SRQ generalization M6 - train-only width sweep

This notebook measures accuracy, persistent state, and analytic-update scaling at widths 2k, 4k, 6k, 8k, and 10k. Widths are reporting points fixed in source, not accuracy-selected candidates. CIFAR-100 test features are never materialized.

In [ ]:
# Edit repository/path values only. Scientific settings and gates are source-locked.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m6_cifar_features'
OUTPUT_DIR='/content/srq_m6_width_sweep_output'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED={
 'configs/srq_generalization_m6_width_sweep_train_only.json':'6c1ca502674e3ede91505308c37f2ed3c884fa54d961ace661f01b36b37bf871',
 'tools/srq_generalization_m6.py':'7ddbd0c1000266c91ef4e6ee517ac3d72f7df2f5860c0ed1c6c35b647a5a6f99',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'methods/analytic_ridge/backends.py':'40114dcc05a9d991682f840efc48e7c74a3e12dde99be049a30ad5f825650dee',
 'methods/analytic_ridge/accounting.py':'a8aca83a0f2f9b2170e54b7739982fa150287939b2b8a7fe06db972196face05',
 'methods/analytic_ridge/compressed_upper.py':'dc3ff6ca1c62255628e40d8a9c29610c5ee138eadf5c352aa3b926c2cacd8f62',
 'methods/analytic_ridge/qr.py':'19d24d887e60ee7fc7adb1a2265253ea261c90636aef241241e5c6faad3a507b',
 'methods/frontends/ranpac.py':'6b94532f607d245c0d148d09c1a36b44dbbf964f4ee9cbf05ba6f64826a90e60'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m6_width_sweep_train_only.json'
RUNNER='tools/srq_generalization_m6.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE LOCK: PASS')

In [ ]:
# Fast local gates before downloading data.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_generalization_m6.py','tests/test_srq_generalization_m5.py','tests/test_srq_generalization_m4.py','tests/test_ranpac_analytic_frontend.py','tests/test_analytic_ridge_backend.py','tests/test_analytic_ridge_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M6 synthetic correctness gate failed; return the complete traceback.'
print('M6 SYNTHETIC GATE: PASS')

In [ ]:
# Download the locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out test features remain absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m6','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run the five locked widths. Preserve diagnostics if a scientific gate fails.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M6 START: widths 2000/4000/6000/8000/10000; Exact, FP16, P2B.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m6_results.json'
assert result_path.is_file(),'M6 failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M6_WIDTH_SWEEP_TRAIN_ONLY','M6 failed; preserve the output and do not relax gates or inspect test accuracy.'

In [ ]:
# Human-readable sweep table.
import pandas as pd
display(pd.read_csv(Path(OUTPUT_DIR)/'width_sweep.csv'))

In [ ]:
# Create source-derived vector figures for the paper.
import matplotlib.pyplot as plt
methods={'exact':'Exact Gram','fp16_square_root':'FP16 square-root','p2b_int8':'P2B INT8/FP32'}
colors={'exact':'#1f77b4','fp16_square_root':'#ff7f0e','p2b_int8':'#2ca02c'}
items=result['width_results']
fig,ax=plt.subplots(figsize=(6.4,4.2))
for method,label in methods.items():
    x=[item['final_total_persistent_bytes'][method]/2**20 for item in items]
    y=[item['validation_aia_percent'][method] for item in items]
    ax.plot(x,y,marker='o',label=label,color=colors[method])
    for state,aia,item in zip(x,y,items): ax.annotate(str(item['width']//1000)+'k',(state,aia),xytext=(3,3),textcoords='offset points',fontsize=8)
ax.set_xscale('log'); ax.set_xlabel('Persistent learner state (MiB, log scale)'); ax.set_ylabel('Validation AIA (%)')
ax.grid(True,alpha=.25); ax.legend(); fig.tight_layout()
state_plot=Path(OUTPUT_DIR)/'width_sweep_accuracy_state.svg'; fig.savefig(state_plot,format='svg'); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.4,4.2))
widths=[item['width'] for item in items]
for method,label in methods.items(): ax.plot(widths,[item['analytic_update_seconds'][method] for item in items],marker='o',label=label,color=colors[method])
ax.set_xlabel('Random-feature width'); ax.set_ylabel('Analytic update time over 10 tasks (s)')
ax.grid(True,alpha=.25); ax.legend(); fig.tight_layout()
time_plot=Path(OUTPUT_DIR)/'width_sweep_update_time.svg'; fig.savefig(time_plot,format='svg'); plt.show(); plt.close(fig)
assert state_plot.is_file() and time_plot.is_file()
print('VECTOR FIGURES READY')

In [ ]:
# Export evidence only; sample-level feature cache and checkpoint are excluded.
bundle=Path('/content/srq_generalization_m6_width_sweep_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for source,name in [(Path(OUTPUT_DIR)/'m6_results.json','m6_results.json'),(Path(OUTPUT_DIR)/'width_sweep.csv','width_sweep.csv'),(Path(OUTPUT_DIR)/'width_sweep_accuracy_state.svg','width_sweep_accuracy_state.svg'),(Path(OUTPUT_DIR)/'width_sweep_update_time.svg','width_sweep_update_time.svg'),(Path(CONFIG),'config.json')]: shutil.copy2(source,bundle/name)
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)